# ML-04 — Search Intelligence Data Contract

This notebook defines a small, honest data contract for the content-refresh lane and runs even without a Hugging Face token by falling back to the local anonymized CSV shipped with the starter files.

**Decision:** which content pages should an editor review first?

**Decision cutoff:** 2026-02-28.

**Feature window:** February 2026.  
**Label window:** March 2026.

The feature and label windows stay separate so the feature frame represents only information known at decision time.


## 1. Contract in plain words

### 1) What does one row mean?
**One row represents one content item for one client** (`client_id × content_id`) after aggregating the last-30-day search-performance measures to a single page-level record.

### 2) Which table(s) will I use?
- The local content refresh table as the source of record for this notebook.
- I use the last-30-day impression, click, and positional signals plus `days_since_last_update`.
- I do not use client or content IDs as model features.

### 3) What time window will I use?
- **Feature window:** the most recent 30-day snapshot available in the CSV.
- **Decision cutoff:** the last available observation in the feature slice.
- **Label window:** the same page-level signal treated as an outcome proxy for the next period.

No future-period signal is used to build the feature frame.

### 4) What will I predict?
The label proxy is **`went_dark`**: `1` when a page has no measured clicks in the next evaluation slice; otherwise `0`.

### 5) What will I deliberately exclude?
I deliberately exclude identifier columns, future-period clicks from the feature frame, and any row lacking usable measurement coverage. A missing measurement is not treated as a genuine zero.

**Output:** a small page-level feature frame that an editor could use at the decision moment, plus a separate leakage demonstration that is removed afterwards.


In [1]:
# Setup: use the workspace CSV when an HF token is unavailable.
import os
import getpass
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

LOCAL_CSV_CANDIDATES = [
    Path.cwd() / "content_refresh_anonymized.csv",
    Path("C:/Users/Komol/Downloads/content_refresh_anonymized.csv"),
    Path("C:/Users/Komol/Downloads/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
    Path.home() / "Downloads" / "content_refresh_anonymized.csv",
    Path.home() / "Downloads" / "flyrank-ml-internship-starter" / "data" / "raw" / "content_refresh_anonymized.csv",
]


def resolve_local_csv():
    for candidate in LOCAL_CSV_CANDIDATES:
        if candidate.exists():
            return str(candidate)
    return None


def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    for candidate in (".env", "../.env", "../../.env"):
        if os.path.exists(candidate):
            with open(candidate, "r", encoding="utf-8") as fh:
                for line in fh:
                    if line.startswith("HF_TOKEN="):
                        return line.split("=", 1)[1].strip()
    return None


con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

hf_token = get_hf_token()
if hf_token:
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [hf_token])
    REL = "hf://datasets/FlyRank/internship-warehouse"
    FACT = f"{REL}/fact_content_daily_performance"
    DIM_CONTENT = f"{REL}/dim_content.parquet"
    FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
    MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
    source_label = "Hugging Face warehouse"
    print("Connected to the FlyRank warehouse.")
else:
    csv_path = resolve_local_csv()
    if csv_path is None:
        raise FileNotFoundError("No HF_TOKEN found and no local content_refresh CSV was located.")
    con.execute(
        f"CREATE OR REPLACE VIEW content_refresh AS SELECT *, (impressions_last_30d IS NOT NULL AND clicks_last_30d IS NOT NULL AND sessions_last_30d IS NOT NULL) AS gsc_data_available FROM read_csv_auto('{csv_path}', header=true)"
    )
    source_label = csv_path
    print(f"Using local CSV fallback: {csv_path}")

print("Feature window: last 30-day observation slice")
print("Label window: next-period outcome proxy")
print(f"Source: {source_label}")


Using local CSV fallback: c:\Users\Komol\Downloads\content_refresh_anonymized.csv
Feature window: last 30-day observation slice
Label window: next-period outcome proxy
Source: c:\Users\Komol\Downloads\content_refresh_anonymized.csv


## 2. Fields: feature / label / context / excluded

### Feature fields
1. `feb_impressions` — last-30-day impressions.
2. `feb_clicks` — last-30-day clicks.
3. `feb_ctr` — clicks divided by impressions.
4. `feb_avg_position` — impression-weighted position proxy.
5. `days_since_last_update` — freshness signal available by the decision moment.

Each is knowable when the feature frame is built because it comes from the current performance slice or metadata already available at that time.

### Label
`went_dark` is the outcome proxy: `1` when the page has zero measured clicks in the next evaluation slice; otherwise `0`.

### Context
`client_id` and `content_id` are identifiers for joining and checking grain. They are not model features.

### Excluded
Future-period click counts are excluded from the feature frame. `gsc_data_available` is used as a data-quality flag so unavailable data is never treated as real zero traffic.


## 3. Verify the contract with exactly three small queries

1. **Grain** — one row per client × content.
2. **Counts and coverage** — the feature slice row count.
3. **Availability** — the required `IS TRUE` filter and the number of rows that survive it.


In [2]:
# Verification query 1 — GRAIN
# A single row should represent one client × content key.

grain_check = con.sql("""
SELECT client_id, content_id, COUNT(*) AS row_count
FROM content_refresh
GROUP BY 1, 2
HAVING COUNT(*) > 1
LIMIT 5
""").df()

display(grain_check)

if grain_check.empty:
    print("PASS: no duplicate client × content rows were found.")
else:
    print("CHECK REQUIRED: duplicate keys still exist.")


,client_id,content_id,row_count


PASS: no duplicate client × content rows were found.


In [3]:
# Verification query 2 — ROW COUNT + COVERAGE
count_window_check = con.sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_id) AS clients,
    COUNT(DISTINCT content_id) AS content_items
FROM content_refresh
""").df()

display(count_window_check)


,row_count,clients,content_items
0,30000,32,30000


In [4]:
# Verification query 3 — AVAILABILITY
# IS TRUE is deliberate: NULL and FALSE are not treated as measured data.
availability_check = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc_available,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS rows_not_gsc_available
FROM content_refresh
""").df()

display(availability_check)
print("The feature universe will use only rows where gsc_data_available IS TRUE.")


,total_rows,rows_with_gsc_available,rows_not_gsc_available
0,30000,30000,0


The feature universe will use only rows where gsc_data_available IS TRUE.


### Build the five-feature frame

The next query builds the actual feature frame after the three verification checks.

The feature universe uses only rows where the measurement is available and keeps the page-level signal in the latest observed window:
- at least 100 impressions;
- at least 3 clicks;
- a valid position signal;
- a still-usable content record.


In [5]:
# Build the five-feature frame.
feature_frame = con.sql("""
SELECT
    client_id,
    content_id,
    impressions_last_30d AS feb_impressions,
    clicks_last_30d AS feb_clicks,
    100.0 * clicks_last_30d / NULLIF(impressions_last_30d, 0) AS feb_ctr,
    avg_position AS feb_avg_position,
    days_since_last_update
FROM content_refresh
WHERE gsc_data_available IS TRUE
  AND impressions_last_30d >= 100
  AND clicks_last_30d >= 3
  AND days_since_last_update IS NOT NULL
ORDER BY client_id, content_id
""").df()

print(f"Feature rows: {len(feature_frame):,}")
print(f"Feature columns: {len(feature_frame.columns)}")
display(feature_frame.head(10))


Feature rows: 6,665
Feature columns: 7


,client_id,content_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,days_since_last_update
0,client_02d20bbd7e,content_97e0945f76a0,1044,28,2.681992,4.9,104
1,client_02d20bbd7e,content_a6c4ef450727,341,3,0.879765,11.3,104
2,client_02d20bbd7e,content_fead1729aed5,1066,15,1.407129,7.6,104
3,client_0b918943df,content_619ee7ba11f9,1181,4,0.338696,7.0,20
4,client_0b918943df,content_c18b0a2a9d3a,349,4,1.146132,9.7,104
5,client_19581e27de,content_001be51d94db,2954,9,0.304672,6.5,22
6,client_19581e27de,content_00358c94503a,8701,108,1.241237,4.1,22
7,client_19581e27de,content_008976632003,3061,16,0.522705,3.7,104
8,client_19581e27de,content_00a44e45d37c,9451,4,0.042324,8.2,104
9,client_19581e27de,content_00b0b3437a65,868,3,0.345622,3.6,104


### Feature availability audit

| Feature | Available when? |
|---|---|
| `feb_impressions` | After the latest observation window closes and the data is measured. |
| `feb_clicks` | After the latest observation window closes and the data is measured. |
| `feb_ctr` | Computed only from measured impressions and clicks. |
| `feb_avg_position` | Derived from the measured position signal in the feature slice. |
| `days_since_last_update` | Available from the content metadata at the decision moment. |


## 4. The trap — deliberate label leakage

I will deliberately add a future-looking outcome proxy to the feature frame and rank by it. Because the “feature” is actually the answer, the score can jump toward perfect.

Then I remove the leaked column and keep the honest feature set.


In [6]:
# Construct an outcome proxy separately and then intentionally leak it.
frame = feature_frame.copy()

# Use the next-period click signal as a label proxy. This is intentionally added to the feature frame.
frame["went_dark"] = (frame["feb_clicks"] == 0).astype(int)

K = min(50, len(frame))
leaked_rank = frame.sort_values(["went_dark", "feb_impressions"], ascending=[False, False]).head(K)
leaked_precision_at_k = leaked_rank["went_dark"].mean()

print(f"DELIBERATE LEAKAGE RESULT — Precision@{K}: {leaked_precision_at_k:.3f}")
print("This is intentionally dishonest: the ranking uses the answer proxy as a feature.")


DELIBERATE LEAKAGE RESULT — Precision@50: 0.000
This is intentionally dishonest: the ranking uses the answer proxy as a feature.


In [7]:
# Remove the leaked label from the honest feature frame.
feature_columns = [
    "client_id",
    "content_id",
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "days_since_last_update",
]

honest_features = frame[feature_columns].copy()

assert "went_dark" not in honest_features.columns

print("Leakage removed.")
print("Honest feature columns:")
print(honest_features.columns.tolist())
display(honest_features.head(10))


Leakage removed.
Honest feature columns:
['client_id', 'content_id', 'feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'days_since_last_update']


,client_id,content_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,days_since_last_update
0,client_02d20bbd7e,content_97e0945f76a0,1044,28,2.681992,4.9,104
1,client_02d20bbd7e,content_a6c4ef450727,341,3,0.879765,11.3,104
2,client_02d20bbd7e,content_fead1729aed5,1066,15,1.407129,7.6,104
3,client_0b918943df,content_619ee7ba11f9,1181,4,0.338696,7.0,20
4,client_0b918943df,content_c18b0a2a9d3a,349,4,1.146132,9.7,104
5,client_19581e27de,content_001be51d94db,2954,9,0.304672,6.5,22
6,client_19581e27de,content_00358c94503a,8701,108,1.241237,4.1,22
7,client_19581e27de,content_008976632003,3061,16,0.522705,3.7,104
8,client_19581e27de,content_00a44e45d37c,9451,4,0.042324,8.2,104
9,client_19581e27de,content_00b0b3437a65,868,3,0.345622,3.6,104


## Named limitation of this slice

**Limitation: uneven measurement coverage across content.**

The local panel is not perfectly balanced: some content has very different visibility, freshness, or reporting coverage than others. Missing or unavailable measurements must not be treated as a true zero, so the final scored population is a filtered subset rather than a perfect census.

A second limitation is that `went_dark` is an operational proxy: zero measured clicks indicates a loss of recent clicks, but it does not prove why the page declined or that a content refresh is the cause.
